##### Import packages

In [1]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pymongo
import nltk
import re

##### Download nltk files

In [3]:
for pkg in ["stopwords", "punkt", "averaged_perceptron_tagger",
            "punkt_tab", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg)
    except Exception:
        pass  # older NLTK versions may not have punkt_tab or averaged_perceptron_tagger_eng

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Liz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Liz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Liz\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Liz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Liz\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


##### Preprocess text in mongoDB

In [4]:
client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["admin"]
tweets = db["Tweets"]

In [5]:
# Preview a few raw texts first (safe to run before defining preprocess)
for doc in tweets.find({'text': {"$exists": True}}).limit(5):
    text = doc["text"]
    print(text)

Train people well enough so they can leave, treat them well enough so they don't want to. —@richardbranson
On Friday evening we welcomed our new MRes and PhD Students on a walking tour of the City. Some of them are International Students and enjoyed seeing the sights of Sydney!  #cityofsydney #macquarieuniexperience @Macquarie_Uni @mqintl @MQSciEng
Excited to publish the BPM Newsletter 1-2018, my first one! Download at https://tinyurl.com/y7ync633  @BPMConf #BPM2018
Our social networks  amplify negative more than the positive. Great article  https://washingtonmonthly.com/magazine/january-february-march-2018/how-to-fix-facebook-before-it-fixes-us/ … cc @webfoundation
Welcome to the official Department of Computing @computing_mq Twitter Account.


In [6]:
text

'Welcome to the official Department of Computing @computing_mq Twitter Account.'

##### Step 1: remove symbols — keep only letters, digits and underscore (\w = [a-zA-Z0-9_])

In [7]:
def step1(text):
    text = re.sub(r"[^a-zA-Z0-9]", " ", text)
    return text

##### a) re.findall: use regular expression (regex) to return all matched characters

In [8]:
# '\w': match [a-zA-Z0-9_]
# + : match 1 or more characters in [a-zA-Z0-9_],  match as many times as possible
# [^...]: match any character not in the set

re.findall(r"[^a-zA-Z0-9]", text)
# re.findall(r"[\w]", text)

[' ', ' ', ' ', ' ', ' ', ' ', ' ', '@', '_', ' ', ' ', '.']

##### b) re.sub: replace match characters

In [9]:
demo_text = "aaaa"

In [10]:
re.sub("a", "b", demo_text)

'bbbb'

In [11]:
res1 = step1(text)   

In [12]:
res1

'Welcome to the official Department of Computing  computing mq Twitter Account '

##### Step 2: Word tokenization

In [13]:
from nltk.tokenize import word_tokenize
def step2(text):
    words_list = word_tokenize(text) # Returns a list like this: ['Welcome', 'to', 'COMP6210']
# print(words_list, " has ", len(words_list), " elements.")
    return words_list

In [14]:
    # clean first, then tokenize
res2 = step2(res1)

In [15]:
res2

['Welcome',
 'to',
 'the',
 'official',
 'Department',
 'of',
 'Computing',
 'computing',
 'mq',
 'Twitter',
 'Account']

##### Step 3： remove Stopwords

In [16]:
from nltk.corpus import stopwords

print(stopwords.words("english"))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [17]:
print(stopwords.words("spanish"))

['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le', 'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin', 'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo', 'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso', 'ante', 'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo', 'otro', 'otras', 'otra', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual', 'poco', 'ella', 'estar', 'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tú', 'te', 'ti', 'tu', 'tus', 'ellas', 'nosotras', 'vosotros', 'vosotras', 'os', 'mío', 'mía', 'míos', 'mías', 'tuyo', 'tuya', 'tuyos', 'tuyas', 'suyo', 'suya', 'suyos', 'suyas', 'nuestro', 'nuestra', 'nuestros', 'nuestras', 'vuestro', 'vuestra', 'vuestros', 'vuestras', 'esos', 'esas', 'estoy', 'estás', 'está', 'estamos', 'estáis', 'están', 'e

In [18]:
# words_list before removing stopwords — notice punctuation is gone after step1
res2

['Welcome',
 'to',
 'the',
 'official',
 'Department',
 'of',
 'Computing',
 'computing',
 'mq',
 'Twitter',
 'Account']

In [19]:
def step3(words_list):
    stop_words = set(stopwords.words("english"))
    results = []
    for word in words_list:
        if word.lower() not in stop_words:
            results.append(word)
    return results

In [20]:
res3 = step3(res2)

In [21]:
res3

['Welcome',
 'official',
 'Department',
 'Computing',
 'computing',
 'mq',
 'Twitter',
 'Account']

##### Step 4: word stemming

In [22]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()
example_words = ["connect", "connections", "connected", "connects", "connection", "connecting", "connectings"]
for word in example_words:
    print(ps.stem(word))

connect
connect
connect
connect
connect
connect
connect


In [23]:
[ps.stem(word) for word in res3]

['welcom', 'offici', 'depart', 'comput', 'comput', 'mq', 'twitter', 'account']

In [24]:
from nltk.stem import PorterStemmer

def step4(results):
    new_results = []
    for word in results:
        new_results.append(ps.stem(word))  # reuse ps defined in Cell 28
    return new_results

res4 = step4(res3)

In [25]:
res4

['welcom', 'offici', 'depart', 'comput', 'comput', 'mq', 'twitter', 'account']

##### Step 5: Pos_tag

In [26]:
def step5(new_results):
    p_tags = nltk.pos_tag(new_results)
    return p_tags
    
res5 = step5(res4)
for tag in res5:
    print(tag)

('welcom', 'NN')
('offici', 'MD')
('depart', 'VB')
('comput', 'NN')
('comput', 'NN')
('mq', 'VBP')
('twitter', 'NN')
('account', 'NN')


##### Process text: 
0. connect to mongodb and extract "text" field; 
1. extract words and numbers; 
2. tokenize text into words; 
3. remove stop words; 
4. stem words; 
5. category words.

In [27]:
def curation(text):
    res1 = step1(text)
    res2 = step2(res1)
    res3 = step3(res2)
    res4 = step4(res3)
    final_res = step5(res4)
    return(final_res)

In [28]:
for docs in tweets.find({'text': {"$exists": True}}).limit(3):
    text = docs["text"]
    processed_text = curation(text)
    print(processed_text)

[('train', 'NN'), ('peopl', 'NN'), ('well', 'RB'), ('enough', 'RB'), ('leav', 'JJ'), ('treat', 'NN'), ('well', 'RB'), ('enough', 'RB'), ('want', 'VBP'), ('richardbranson', 'NN')]
[('friday', 'RB'), ('even', 'RB'), ('welcom', 'VBP'), ('new', 'JJ'), ('mre', 'NN'), ('phd', 'NN'), ('student', 'NN'), ('walk', 'VBP'), ('tour', 'NN'), ('citi', 'NN'), ('intern', 'JJ'), ('student', 'NN'), ('enjoy', 'NN'), ('see', 'VBP'), ('sight', 'JJ'), ('sydney', 'NN'), ('cityofsydney', 'NN'), ('macquarieuniexperi', 'NN'), ('macquari', 'NN'), ('uni', 'JJ'), ('mqintl', 'NN'), ('mqscieng', 'NN')]
[('excit', 'JJ'), ('publish', 'JJ'), ('bpm', 'NN'), ('newslett', 'RB'), ('1', 'CD'), ('2018', 'CD'), ('first', 'JJ'), ('one', 'CD'), ('download', 'NN'), ('http', 'NN'), ('tinyurl', 'NN'), ('com', 'NN'), ('y7ync633', 'NN'), ('bpmconf', 'NN'), ('bpm2018', 'NN')]


##### Question 1

In [29]:
# reuse the connection from Cell 6 (client / tweets)
sample_cursor = tweets.find({"text": {"$exists": True}}, {"text": 1}).limit(5)
for i, doc in enumerate(sample_cursor, start=1):
    categorized_words = curation(doc["text"])
    print(f"[sample {i}]", categorized_words[:10])
    print()

[sample 1] [('train', 'NN'), ('peopl', 'NN'), ('well', 'RB'), ('enough', 'RB'), ('leav', 'JJ'), ('treat', 'NN'), ('well', 'RB'), ('enough', 'RB'), ('want', 'VBP'), ('richardbranson', 'NN')]

[sample 2] [('friday', 'RB'), ('even', 'RB'), ('welcom', 'VBP'), ('new', 'JJ'), ('mre', 'NN'), ('phd', 'NN'), ('student', 'NN'), ('walk', 'VBP'), ('tour', 'NN'), ('citi', 'NN')]

[sample 3] [('excit', 'JJ'), ('publish', 'JJ'), ('bpm', 'NN'), ('newslett', 'RB'), ('1', 'CD'), ('2018', 'CD'), ('first', 'JJ'), ('one', 'CD'), ('download', 'NN'), ('http', 'NN')]

[sample 4] [('social', 'JJ'), ('network', 'NN'), ('amplifi', 'JJ'), ('neg', 'JJ'), ('posit', 'NN'), ('great', 'JJ'), ('articl', 'JJ'), ('http', 'NN'), ('washingtonmonthli', 'NN'), ('com', 'NN')]

[sample 5] [('welcom', 'NN'), ('offici', 'MD'), ('depart', 'VB'), ('comput', 'NN'), ('comput', 'NN'), ('mq', 'VBP'), ('twitter', 'NN'), ('account', 'NN')]



In [30]:
# categorized_words comes from the last iteration of Cell 38
result = ""
for word, tag in categorized_words:
    result = result + word + "," + tag + "\n"

In [31]:
result

'welcom,NN\noffici,MD\ndepart,VB\ncomput,NN\ncomput,NN\nmq,VBP\ntwitter,NN\naccount,NN\n'

In [32]:
print(result)

welcom,NN
offici,MD
depart,VB
comput,NN
comput,NN
mq,VBP
twitter,NN
account,NN



#### Question 2 steps: 
0. extract "text" of each document; 
1. obtain the category of words; 
2. transform to comma-separated value (CSV) format; 
3. update document.

 ##### In a CSV file, each line represents a single row of data, and the values within each row are separated by commas.

In [33]:
# reuse the connection from Cell 6 (tweets)
def categorized_words_2_csv(categorized_words):
    return "\n".join([f"{word},{tag}" for word, tag in categorized_words])


# Run a small batch first in class; remove .limit(...) for full processing
for doc in tweets.find({"text": {"$exists": True}}, {"text": 1}).limit(20):
    # Step 0 & 1
    categorized_words = curation(doc["text"])

    # Step 2
    csv = categorized_words_2_csv(categorized_words)

    # Step 3
    tweets.update_one(
        {"_id": doc["_id"]},
        {
            "$set": {
                "keyword": csv,
                "processor_version": "week5_optimized"
            }
        }
    )